# 🧹 Module 3 — Class 1: Clean the Customer Data

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 The story

- Monday morning. You work at a telecom in Tashkent.
- **Boss:** *"Tell me who will leave next month."*
- **Data team:** *"Here's `customers.csv`. We merged 3 systems. It's a mess. Fix it."*

**Today we clean. Tomorrow we train.**

## 💡 Why does this matter?

- Bad data → wrong predictions → angry boss
- Real data is **always** messy
- Cleaning = 80% of a data scientist's work

## 🎯 Today's plan

1. **Look** at the data
2. **Fix** the money column
3. **Fill** empty values
4. **Same word** for same thing
5. **Remove** duplicate customers
6. **Fix** weird numbers
7. **Save** the clean file
8. **Bonus:** apply to real Telco data

## 🤖 How to run

Click a code cell → press **Shift + Enter**.

Read the comments (lines with `#`). They explain what each line does.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
print('✅ Tools ready')

---
## Step 1: Get the file 📥

We make a fake (but realistic) dirty file. Real customer data is private — we can't share it.

**Just run the next cell.** Don't worry about how it works.

In [ ]:
# 🔒 Don't read this. Just run it. It makes a dirty customer file.
def make_dirty_customers():
    rng = np.random.default_rng(42)
    n = 3000
    names  = ['Ali','Zara','Jasur','Madina','Otabek','Aisha','Karim','Nargiza',
              'Bobur','Dilnoza','Sardor','Malika','Timur','Shaxnoza','Rustam','Feruza']
    cities = ['Tashkent','Samarkand','Bukhara','Andijan','Namangan','Fergana']
    plans  = ['basic','pro','premium']
    nets   = ['Fiber','DSL','No internet service']
    df = pd.DataFrame({
        'customer_id':       [f'CU{i:05d}' for i in range(1, n+1)],
        'name':              rng.choice(names, n),
        'city':              rng.choice(cities, n),
        'age':               rng.integers(18, 75, n).astype(float),
        'tenure_months':     rng.integers(1, 73, n).astype(float),
        'monthly_charges':   rng.normal(80, 25, n).round(2),
        'plan':              rng.choice(plans, n),
        'internet_service':  rng.choice(nets, n, p=[0.45,0.35,0.20]),
        'has_phone_service': rng.choice(['Yes','No phone service'], n, p=[0.9,0.1]),
        'churn':             rng.choice(['Yes','No'], n, p=[0.27,0.73])
    })
    df['total_charges'] = (df['tenure_months'] * df['monthly_charges']).round(2)
    # add real-world dirt:
    df.loc[rng.choice(n, 80, replace=False), 'tenure_months']  = np.nan
    df.loc[rng.choice(n, 60, replace=False), 'monthly_charges']= np.nan
    df.loc[rng.choice(n, 40, replace=False), 'age']            = np.nan
    df['total_charges'] = df['total_charges'].astype(str)
    df.loc[rng.choice(n, 25, replace=False), 'total_charges'] = ' '
    ti = df.index[df['city']=='Tashkent'].tolist(); rng.shuffle(ti)
    df.loc[ti[:80],'city']='tashkent'; df.loc[ti[80:140],'city']='TASHKENT'
    df.loc[ti[140:200],'city']=' Tashkent '
    pi = df.index[df['plan']=='pro'].tolist(); rng.shuffle(pi)
    df.loc[pi[:50],'plan']='PRO'; df.loc[pi[50:80],'plan']='Pro '
    yi = df.index[df['churn']=='Yes'].tolist(); rng.shuffle(yi)
    df.loc[yi[:30],'churn']='yes'; df.loc[yi[30:60],'churn']='YES'
    df.loc[yi[60:90],'churn']=' Yes '
    df.loc[100,'tenure_months']=-3; df.loc[200,'tenure_months']=999
    df.loc[300,'monthly_charges']=99999; df.loc[400,'age']=200
    df = pd.concat([df, df.iloc[[5,50,100,500,1000,2000]]], ignore_index=True)
    return df

df = make_dirty_customers()
print(f'📄 Got file: {len(df)} rows')

In [ ]:
# Look at first 5 rows
df.head()

👀 Looks fine? **Trust me — it's not.** Let's find the problems.

Each row = one customer. Important columns:
- `tenure_months` — how long they've been with us
- `monthly_charges` — what they pay each month
- `churn` — did they leave? (Yes / No) ← *what we want to predict*

---
## Step 2: Look 👀

> **Always look before you change.** A doctor checks before they cut.

Three commands tell us a lot:

In [ ]:
# Show types and how many values are NOT empty
df.info()

🚨 **Problem 1 found!**

- `total_charges` is `object` (text). But it's money — should be a number!
- `tenure_months`, `monthly_charges`, `age` have empty values

Types you'll see:
- `int64` — whole number (5)
- `float64` — number with a dot (19.95)
- `object` / `str` — text

In [ ]:
# Show numbers about number columns
df.describe()

🚨 **Problem 2: weird numbers!**

Look at `min` and `max`:
- `tenure_months` min = **−3** ❌ (negative!)
- `tenure_months` max = **999** ❌ (83 years!)
- `monthly_charges` max = **99,999** ❌ (typo? VIP?)
- `age` max = **200** ❌

👉 Always check `min` and `max`. Bad data hides there.

In [ ]:
# Count empty values per column
df.isnull().sum()

In [ ]:
# Look at city column
df['city'].value_counts()

🚨 **Problem 3: 4 spellings of Tashkent!**

- `Tashkent` (290)
- `tashkent` (80) ← signup form
- `TASHKENT` (60) ← phone keyboard
- `' Tashkent '` (60) ← Excel

Model thinks: 4 different cities. Wrong!

---
## Step 3: Fix the money column 💰

> **Why?** Math doesn't work on text. `"5" + "3" = "53"` (wrong).

Find the bad rows first.

In [ ]:
# Find rows that aren't real numbers
bad = pd.to_numeric(df['total_charges'], errors='coerce').isna()
print(f'Bad rows: {bad.sum()}')
df.loc[bad, ['customer_id', 'total_charges']].head()

💡 **AHA!** They contain `' '` (a blank space). That's why pandas reads everything as text.

**Fix it:**

In [ ]:
# Convert to number (bad becomes NaN)
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# Fill NaN with median
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

print('Type now:', df['total_charges'].dtype)
print('Empty:', df['total_charges'].isna().sum())

✅ Done. The money column is now a real number column.

---
## Step 4: Fill empty values 🕳️

> **Why?** Models crash on empty values.

Three ways to fill:

| Way | Best for |
| --- | --- |
| **Mean** (average) | Numbers, no big outliers |
| **Median** (middle) | Numbers, **safer** with outliers |
| **Mode** (most common) | Text columns |

In [ ]:
# Show empty values per column
df.isnull().sum()

In [ ]:
# Fill 3 columns with their median
for col in ['tenure_months', 'monthly_charges', 'age']:
    df[col] = df[col].fillna(df[col].median())

print('Empty values now:', df.isnull().sum().sum())

✅ Zero empty values. Why median? Because we have a 99,999 outlier — mean would be wrong.

### 📝 YOUR TURN — make a 'was empty' column

Sometimes 'this was empty' is useful info!

**But you must do it BEFORE filling.** Once filled, `isna()` returns all False.

Hint: re-load with `df_raw = make_dirty_customers()`, then:
```python
df_raw['tenure_was_missing'] = df_raw['tenure_months'].isna().astype(int)
```

In [ ]:
# YOUR CODE here


---
## Step 5: Same word, same meaning 🧽

> **Why?** `'Yes'` and `'yes'` and `' Yes '` all mean Yes. Model thinks 3 categories.

**The 2-step pattern:**
```python
df['col'].str.strip().str.lower()
```
- `.str.strip()` removes spaces
- `.str.lower()` makes lowercase

In [ ]:
# Before
print(df['city'].value_counts())

In [ ]:
# Clean 3 text columns
for col in ['city', 'plan', 'churn']:
    df[col] = df[col].str.strip().str.lower()

# After
print(df['city'].value_counts())

✅ One spelling per city now.

Now fix verbose values: `'No phone service'` → `'No'`.

In [ ]:
# 'No phone service' really means 'No'
df['has_phone_service'] = df['has_phone_service'].replace('No phone service', 'No')
df['has_phone_service'].unique()

---
## Step 6: Remove duplicate customers 👯

> **Why?** Same customer logged twice = wrong metrics. Model 'memorizes' that customer.

Use `customer_id` as the key (not the whole row).

In [ ]:
# Count duplicates
print('Duplicates:', df.duplicated(subset=['customer_id']).sum())

# Drop them. Keep first.
df = df.drop_duplicates(subset=['customer_id']).reset_index(drop=True)

print('Rows now:', len(df))

---
## Step 7: Fix weird numbers 🚨

Three kinds of outliers:

| Kind | Example | What to do |
| --- | --- | --- |
| **Mistake** | tenure = -3 | Remove or fix |
| **Real but big** | VIP customer | Keep, maybe cap |
| **Fraud** | wants to be detected | **Never** remove |

🚨 **Always think first!**

In [ ]:
# Step 1: fix IMPOSSIBLE values
bad_tenure = (df['tenure_months'] < 0) | (df['tenure_months'] > 120)
bad_age    = (df['age'] < 0) | (df['age'] > 120)

print('Bad tenure:', df.loc[bad_tenure, 'tenure_months'].tolist())
print('Bad age   :', df.loc[bad_age, 'age'].tolist())

In [ ]:
# Replace impossible with NaN, then fill with median
df.loc[bad_tenure, 'tenure_months'] = np.nan
df.loc[bad_age,    'age']           = np.nan
df['tenure_months'] = df['tenure_months'].fillna(df['tenure_months'].median())
df['age']           = df['age'].fillna(df['age'].median())
print('✅ Fixed')

**Now find statistical outliers (IQR rule):**

```
Q1 = 25% point, Q3 = 75% point, IQR = Q3 - Q1
low  = Q1 - 1.5 × IQR
high = Q3 + 1.5 × IQR
outside [low, high] = outlier
```

In [ ]:
# Find outliers in monthly_charges
Q1 = df['monthly_charges'].quantile(0.25)
Q3 = df['monthly_charges'].quantile(0.75)
IQR = Q3 - Q1
low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR

outliers = (df['monthly_charges'] < low) | (df['monthly_charges'] > high)
print(f'Outliers: {outliers.sum()}  (limits {low:.1f} to {high:.1f})')

In [ ]:
# Cap (winsorize) — keep customer, prevent extreme value
df['monthly_charges'] = df['monthly_charges'].clip(lower=low, upper=high)
new_max = df['monthly_charges'].max()
print(f'Max now: {new_max:.2f}')

In [ ]:
# Visualize with box plots
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for i, col in enumerate(['monthly_charges', 'total_charges', 'tenure_months']):
    ax[i].boxplot(df[col].dropna())
    ax[i].set_title(col)
plt.tight_layout(); plt.show()

💡 **Lesson:** Big number ≠ wrong number. A 72-month customer is loyal, not an error.

---
## Step 8: Save 💾

> **Rule:** Never overwrite the original file. Save with a NEW name.

In [ ]:
# Final check
print('Shape:', df.shape)
print('Empty:', df.isnull().sum().sum())
df.dtypes.value_counts()

In [ ]:
# Save
df.to_csv('customers_cleaned.csv', index=False)
print('✅ Saved customers_cleaned.csv')

---
## 🎁 Bonus: Apply to real data

We just cleaned synthetic data. **Same techniques work on real public datasets.**

We'll use **Telco Customer Churn** (a real public dataset). It's mostly clean but has 2 real problems we know how to fix.

👉 **Output of this section is what we use in Class 2 to train a model!**

In [ ]:
# Load real Telco data
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
telco = pd.read_csv(url)
print(f'Loaded {len(telco)} customers')

Two problems in Telco (the same ones we solved!):
1. `TotalCharges` is text → fix it
2. `'No internet service'` / `'No phone service'` → collapse to `'No'`

In [ ]:
# Fix 1: TotalCharges as text
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce').fillna(0)

# Fix 2: collapse verbose values
verbose_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
for c in verbose_cols:
    telco[c] = telco[c].replace('No internet service', 'No')
telco['MultipleLines'] = telco['MultipleLines'].replace('No phone service', 'No')

# Save for Class 2
telco.to_csv('telco_churn_cleaned.csv', index=False)
print('✅ Saved telco_churn_cleaned.csv')
print('TotalCharges type:', telco['TotalCharges'].dtype)

---
## 🏁 We did it!

### What we cleaned today

| Step | Found | Fixed |
| --- | --- | --- |
| 2. Look | text where number should be, empty values, weird numbers | (just looking) |
| 3. Money | `' '` blanks in `total_charges` | text → number, fill |
| 4. Empty | 180 missing values | filled with median |
| 5. Words | 4 spellings of Tashkent | one spelling each |
| 6. Copies | 6 duplicates from merge | dropped |
| 7. Weird | tenure -3, 999, age 200, charges 99999 | replaced + capped |
| 8. Save | — | `customers_cleaned.csv` |
| Bonus | 2 real Telco problems | `telco_churn_cleaned.csv` |

### 🎯 Where we go next

- **Class 2:** Encoding + scaling (prepare for model)
- **Class 3:** Feature engineering (build new columns)
- **Module 4:** Train the model. Predict who will leave!

### 🎓 Words to remember

- **NaN** — empty value
- **dtype** — type of a column
- **Median** — middle value (safer than mean)
- **Outlier** — value far from the others
- **Clip / winsorize** — cap to a max

### 📤 Submit

1. Save as `Module3_Class1_<YourName>.ipynb`
2. PR to your group repo at `module-3/class_1/submissions/<YourName>/`

🧹 *Good job. See you in Class 2!*